# PLD_accounting Tutorial: Random Allocation Accounting in DP-SGD Language

This notebook provides several examples for PLD accounting of DP-SGD using random allocation.

Each example starts from natural DP-SGD inputs `(sample_size=n, batch_size=b, epochs=E)` and then derives the random-allocation inputs expected by `PLD_accounting`.

In the standard DP-SGD interpretation used here, we translate to `num_steps = int(n * E // b)`, `num_selected = E`, and `num_epochs = 1`.

This means `num_steps` is the total number of optimizer steps across training, while `num_selected` records that each example is expected to be selected once per epoch over `E` epochs.


## Installation

Uncomment the next cell if you want to install the package into the notebook environment.


In [ ]:
# !pip install PLD_accounting

In [ ]:
import numpy as np
from IPython.display import Math, display
from PLD_accounting import (
    DEFAULT_TAIL_TRUNCATION,
    AllocationSchemeConfig,
    BoundType,
    ConvolutionMethod,
    PrivacyParams,
    gaussian_allocation_delta_configurable,
    gaussian_allocation_epsilon_configurable,
    gaussian_allocation_epsilon_range,
    gaussian_allocation_PLD,
    gaussian_distribution,
    general_allocation_delta,
    general_allocation_epsilon,
    general_allocation_PLD,
    laplace_distribution,
    subsample_PLD,
)


## Example 1: Adaptive Gaussian epsilon bounds

`gaussian_allocation_epsilon_range` is the quickest way to get an upper and lower bound on $\varepsilon$ for a DP-SGD-style configuration.

The code below translates the DP-SGD parameters to the random-allocation language and then provides a $\varepsilon$ interval.

In [ ]:
sample_size = 10_000
batch_size = 50
dp_sgd_epochs = 5
noise_multiplier = 1.0
target_delta = 1e-8
epsilon_accuracy = 0.1

num_steps = int(sample_size * dp_sgd_epochs // batch_size)
num_selected = dp_sgd_epochs
allocation_num_epochs = 1

epsilon_upper, epsilon_lower = gaussian_allocation_epsilon_range(
    delta=target_delta,
    sigma=noise_multiplier,
    num_steps=num_steps,
    num_selected=num_selected,
    num_epochs=allocation_num_epochs,
    epsilon_accuracy=epsilon_accuracy,
)

display(Math(rf"""\Large \begin{{aligned}}
&\text{{A DP-SGD process with random allocation sampling of }} {sample_size:,} \text{{ elements for }} {dp_sgd_epochs} \text{{ epochs with expected batch size }} {batch_size:,} \\
&\text{{ using the Gaussian mechanism with }} \sigma = {noise_multiplier}, \text{{ is }} (\varepsilon, \delta)\text{{-DP for }} \delta = \text{{{target_delta:.0e}}} \text{{ and }} \varepsilon \in [{epsilon_lower:.4f}, {epsilon_upper:.4f}]
\end{{aligned}}"""))


## Example 2: Configurable Gaussian one-sided queries

`PrivacyParams` and `AllocationSchemeConfig` let you control discretization, tail truncation, the convolution backend, and the direction of the bound.

#### Example 2.1 computes an upper bound on $\varepsilon$ at fixed $\delta$.


In [ ]:
# Example 2.1: configurable upper bound on epsilon
sample_size = 10_000
batch_size = 500
dp_sgd_epochs = 5
noise_multiplier = 3.0
target_delta = 1e-6

num_steps = int(sample_size * dp_sgd_epochs // batch_size)
num_selected = dp_sgd_epochs
allocation_num_epochs = 1

epsilon_upper_params = PrivacyParams(
    sigma=noise_multiplier,
    num_steps=num_steps,
    num_selected=num_selected,
    num_epochs=allocation_num_epochs,
    delta=target_delta,
)
epsilon_upper_config = AllocationSchemeConfig(
    loss_discretization=0.05,
    tail_truncation=target_delta / 10,
    max_grid_mult=50_000,
    convolution_method=ConvolutionMethod.BEST_OF_TWO,
)
epsilon_upper_configurable = gaussian_allocation_epsilon_configurable(
    params=epsilon_upper_params,
    config=epsilon_upper_config,
    bound_type=BoundType.DOMINATES,
)

display(Math(rf"""\Large \begin{{aligned}}
&\text{{A DP-SGD process with random allocation sampling of }} {sample_size:,} \text{{ elements for }} {dp_sgd_epochs} \text{{ epochs with expected batch size }} {batch_size:,} \\n
&\text{{ using the Gaussian mechanism with }} \sigma = {noise_multiplier}, \text{{ has configurable upper bound }} \varepsilon \le {epsilon_upper_configurable:.4f} \text{{ at }} \delta = \text{{{target_delta:.0e}}}
\end{{aligned}}"""))


#### Example 2.2 computes a lower bound on $\delta$ at fixed $\varepsilon$.


In [ ]:
# Example 2.2: configurable lower bound on delta
sample_size = 10_000
batch_size = 500
dp_sgd_epochs = 5
noise_multiplier = 3.0
target_epsilon = 0.8
reference_delta = 1e-6

num_steps = int(sample_size * dp_sgd_epochs // batch_size)
num_selected = dp_sgd_epochs
allocation_num_epochs = 1

delta_lower_params = PrivacyParams(
    sigma=noise_multiplier,
    num_steps=num_steps,
    num_selected=num_selected,
    num_epochs=allocation_num_epochs,
    epsilon=target_epsilon,
)
delta_lower_config = AllocationSchemeConfig(
    loss_discretization=0.05,
    tail_truncation=reference_delta / 10,
    max_grid_mult=50_000,
    convolution_method=ConvolutionMethod.GEOM,
)
delta_lower_configurable = gaussian_allocation_delta_configurable(
    params=delta_lower_params,
    config=delta_lower_config,
    bound_type=BoundType.IS_DOMINATED,
)
delta_lower_text = f"{delta_lower_configurable:.2e}" if delta_lower_configurable else "0"

display(Math(rf"""\Large \begin{{aligned}}
&\text{{A DP-SGD process with random allocation sampling of }} {sample_size:,} \text{{ elements for }} {dp_sgd_epochs} \text{{ epochs with expected batch size }} {batch_size:,} \\n
&\text{{ using the Gaussian mechanism with }} \sigma = {noise_multiplier}, \text{{ has configurable lower bound }} \delta \ge \text{{{delta_lower_text}}} \text{{ at }} \varepsilon = {target_epsilon:.1f}
\end{{aligned}}"""))


## Example 3: Gaussian full PLD object

`gaussian_allocation_PLD` builds a reusable `dp_accounting` PLD upper bound using `BoundType.DOMINATES`.


In [ ]:
# Example 3.1: construct a reusable upper-bound Gaussian PLD
sample_size = 10_000
batch_size = 500
dp_sgd_epochs = 5
noise_multiplier = 3.0
target_delta = 1e-6

num_steps = int(sample_size * dp_sgd_epochs // batch_size)
num_selected = dp_sgd_epochs
allocation_num_epochs = 1

gaussian_pld_upper_params = PrivacyParams(
    sigma=noise_multiplier,
    num_steps=num_steps,
    num_selected=num_selected,
    num_epochs=allocation_num_epochs,
    delta=target_delta,
)
gaussian_pld_upper_config = AllocationSchemeConfig(
    loss_discretization=0.05,
    tail_truncation=1e-7,
    max_grid_mult=50_000,
    convolution_method=ConvolutionMethod.BEST_OF_TWO,
)
gaussian_pld_upper = gaussian_allocation_PLD(
    params=gaussian_pld_upper_params,
    config=gaussian_pld_upper_config,
    bound_type=BoundType.DOMINATES,
)

display(Math(rf"""\Large \begin{{aligned}}
&\text{{An upper-bound Gaussian PLD for a DP-SGD process with random allocation sampling of }} {sample_size:,} \text{{ elements for }} {dp_sgd_epochs} \text{{ epochs}}\\
&\text{{with expected batch size }} {batch_size:,} \text{{ and Gaussian noise }} \sigma = {noise_multiplier} \text{{ has been constructed using }} \texttt{{BoundType.DOMINATES}}
\end{{aligned}}"""))


The following cell then queries several $(\varepsilon, \delta)$ pairs from that same upper-bound PLD.

In [ ]:
# Example 3.2: query epsilon and delta from the reusable upper-bound PLD
delta_query_values = [1e-4, 1e-5, 1e-6]
epsilon_query_values = [0.7, 0.8, 0.9]

epsilon_query_results = [
    gaussian_pld_upper.get_epsilon_for_delta(delta_value)
    for delta_value in delta_query_values
]
delta_query_results = [
    gaussian_pld_upper.get_delta_for_epsilon(epsilon_value)
    for epsilon_value in epsilon_query_values
]

display(Math(rf"""\Large \begin{{aligned}}
&\text{{The reusable upper-bound Gaussian PLD gives }} \varepsilon = {epsilon_query_results[0]:.4f} \text{{ at }} \delta = \text{{{delta_query_values[0]:.0e}}}, \varepsilon = {epsilon_query_results[1]:.4f} \text{{ at }} \delta = \text{{{delta_query_values[1]:.0e}}},\\
&\varepsilon = {epsilon_query_results[2]:.4f} \text{{ at }} \delta = \text{{{delta_query_values[2]:.0e}}}, \delta = \text{{{delta_query_results[0]:.2e}}} \text{{ at }} \varepsilon = {epsilon_query_values[0]:.1f},\\
&\delta = \text{{{delta_query_results[1]:.2e}}} \text{{ at }} \varepsilon = {epsilon_query_values[1]:.1f}, \text{{ and }} \delta = \text{{{delta_query_results[2]:.2e}}} \text{{ at }} \varepsilon = {epsilon_query_values[2]:.1f}
\end{{aligned}}"""))


## Example 4: Realization-based allocation APIs

`general_allocation_PLD`, `general_allocation_epsilon`, and `general_allocation_delta` work with explicit loss-space realizations.

#### 4.1: Build Gaussian realizations.

In [ ]:
# Example 4.1: construct Gaussian realizations (native mechanism discretization)
noise_multiplier = 3.0
num_grid_points = 2200
num_std = 8.0
loss_std = 1.0 / noise_multiplier
value_discretization = (2.0 * num_std * loss_std) / (num_grid_points - 1)

gaussian_remove_realization = gaussian_distribution(
    noise_multiplier,
    value_discretization=value_discretization,
    tail_truncation=DEFAULT_TAIL_TRUNCATION,
)
gaussian_add_realization = gaussian_remove_realization.copy()

display(Math(rf"""\Large \begin{{aligned}}
&\text{{Gaussian REMOVE and ADD realizations built from the Gaussian mechanism with }} \sigma = {noise_multiplier} \text{{ each have support size }} {gaussian_remove_realization.prob_arr.size}
\end{{aligned}}"""))


#### 4.2: Build Laplace realizations.

In [ ]:
# Example 4.2: construct Laplace realizations (native mechanism discretization)
def laplace_scale_from_noise_std(noise_std: float) -> float:
    return float(noise_std) / np.sqrt(2.0)

noise_multiplier = 3.0
laplace_discretization = 5e-3

laplace_remove_realization = laplace_distribution(
    laplace_scale_from_noise_std(noise_multiplier),
    value_discretization=laplace_discretization,
    tail_truncation=DEFAULT_TAIL_TRUNCATION,
)
laplace_add_realization = laplace_remove_realization.copy()

display(Math(rf"""\Large \begin{{aligned}}
&\text{{Laplace REMOVE and ADD realizations built from the Laplace mechanism with noise standard deviation }} {noise_multiplier} \\n
&\text{{have support sizes }} {laplace_remove_realization.prob_arr.size} \text{{ and }} {laplace_add_realization.prob_arr.size}
\end{{aligned}}"""))


#### 4.3: Use the Gaussian realization to call all three APIs on the same DP-SGD example.

Replacing it with the Laplace realization uses the same pattern.

In [ ]:
# Example 4.3: call general_allocation_PLD, general_allocation_epsilon, and general_allocation_delta
sample_size = 10_000
batch_size = 500
dp_sgd_epochs = 5
noise_multiplier = 3.0
target_delta = 1e-6
target_epsilon = 0.8

num_steps = int(sample_size * dp_sgd_epochs // batch_size)
num_selected = dp_sgd_epochs
allocation_num_epochs = 1

# Swap these two lines to laplace_remove_realization / laplace_add_realization
# to run the same realization-based example for Laplace.
selected_mechanism_name = "Gaussian"
selected_remove_realization = gaussian_remove_realization
selected_add_realization = gaussian_add_realization

realization_upper_config = AllocationSchemeConfig(
    loss_discretization=5e-3,
    tail_truncation=1e-10,
    max_grid_mult=30_000,
    convolution_method=ConvolutionMethod.GEOM,
)
realization_upper_pld = general_allocation_PLD(
    num_steps=num_steps,
    num_selected=num_selected,
    num_epochs=allocation_num_epochs,
    remove_realization=selected_remove_realization,
    add_realization=selected_add_realization,
    config=realization_upper_config,
    bound_type=BoundType.DOMINATES,
)
epsilon_from_realization_pld = realization_upper_pld.get_epsilon_for_delta(target_delta)
epsilon_from_realization_api = general_allocation_epsilon(
    delta=target_delta,
    num_steps=num_steps,
    num_selected=num_selected,
    num_epochs=allocation_num_epochs,
    remove_realization=selected_remove_realization,
    add_realization=selected_add_realization,
    config=realization_upper_config,
    bound_type=BoundType.DOMINATES,
)
delta_from_realization_api = general_allocation_delta(
    epsilon=target_epsilon,
    num_steps=num_steps,
    num_selected=num_selected,
    num_epochs=allocation_num_epochs,
    remove_realization=selected_remove_realization,
    add_realization=selected_add_realization,
    config=realization_upper_config,
    bound_type=BoundType.DOMINATES,
)
delta_from_realization_text = f"{delta_from_realization_api:.2e}" if delta_from_realization_api else "0"

display(Math(rf"""\Large \begin{{aligned}}
&\text{{A DP-SGD process with random allocation sampling of }} {sample_size:,} \text{{ elements for }} {dp_sgd_epochs} \text{{ epochs with expected batch size }} {batch_size:,} \\n
&\text{{using the }} {selected_mechanism_name} \text{{ realization with }} \texttt{{BoundType.DOMINATES}} \text{{ gives upper bound }} \varepsilon = {epsilon_from_realization_pld:.4f} \text{{ at }} \delta = \text{{{target_delta:.0e}}} \text{{ from }} \texttt{{general\_allocation\_PLD}},\\
&\text{{upper bound }} \varepsilon = {epsilon_from_realization_api:.4f} \text{{ from }} \texttt{{general\_allocation\_epsilon}} \text{{ and upper bound }} \delta = \text{{{delta_from_realization_text}}} \text{{ from }} \texttt{{general\_allocation\_delta}} \text{{ at }} \varepsilon = {target_epsilon:.1f}
\end{{aligned}}"""))


## Example 5: Combining construction, subsampling, and composition

`subsample_PLD` lets you take a reusable PLD, amplify it by subsampling, and then compose it.

The code below shows one way to combine Gaussian PLD construction, subsampling, and repeated composition in a single workflow.


In [ ]:
sample_size = 10_000
batch_size = 500
dp_sgd_epochs = 5
noise_multiplier = 3.0
target_delta = 1e-6

num_rounds = int(sample_size * dp_sgd_epochs // batch_size)
client_sampling_probability = batch_size / sample_size
per_round_num_steps = int(num_rounds // dp_sgd_epochs)

round_config = AllocationSchemeConfig(
    loss_discretization=0.05,
    tail_truncation=(target_delta / 10) / (num_rounds * client_sampling_probability),
    max_grid_mult=50_000,
    convolution_method=ConvolutionMethod.BEST_OF_TWO,
)
round_pld_upper = gaussian_allocation_PLD(
    params=PrivacyParams(
        sigma=noise_multiplier,
        num_steps=per_round_num_steps,
        num_selected=1,
        num_epochs=1,
        delta=target_delta,
    ),
    config=round_config,
    bound_type=BoundType.DOMINATES,
)
combined_pld_upper = subsample_PLD(
    pld=round_pld_upper,
    sampling_probability=client_sampling_probability,
).self_compose(num_rounds)
final_epsilon = combined_pld_upper.get_epsilon_for_delta(target_delta)

display(Math(rf"""\Large \begin{{aligned}}
&\text{{Combining Gaussian PLD construction, subsampling with }} q = {client_sampling_probability:.4f}, \text{{ and }} {num_rounds} \text{{ rounds of composition}}\\
&\text{{for a DP-SGD process with random allocation sampling of }} {sample_size:,} \text{{ elements for }} {dp_sgd_epochs} \text{{ epochs and expected batch size }} {batch_size:,} \\n
&\text{{gives }} (\varepsilon, \delta)\text{{-DP for }} \delta = \text{{{target_delta:.0e}}} \text{{ and }} \varepsilon = {final_epsilon:.4f}
\end{{aligned}}"""))


## Covered Public Functions

This notebook includes runnable examples for the following public functions:

- `gaussian_allocation_epsilon_range`
- `gaussian_allocation_epsilon_configurable`
- `gaussian_allocation_delta_configurable`
- `gaussian_allocation_PLD`
- `general_allocation_PLD`
- `general_allocation_epsilon`
- `general_allocation_delta`
- `gaussian_distribution`
- `laplace_distribution`
- `subsample_PLD`
